# Mask2Former — U-DIADS-TL tiled 1152 on Colab

Fine-tunes Hugging Face `facebook/mask2former-swin-tiny-coco-instance` on
`Latin14396`, `Latin2`, and `Syr341`, then reports the repository's Zottin
instance metrics on every test page. The recipe deliberately matches the other
Colab experiments where it is meaningful: **maximum image side 1152**, batch 1,
AdamW, seed 42, validation-FM checkpoint selection, and AMP.

Mask2Former preserves the page aspect ratio rather than distorting or letterboxing to
a square. Latin14396 uses the full page; Latin2 and Syr341 use two and three vertical
tiles respectively. Every input keeps a 1152-pixel maximum side, while tiling reduces
the number of lines assigned to 100 learned queries and multiplies the training samples.
At inference, horizontally aligned fragments are stitched across tile seams; only
instance IDs are merged, while mask pixels remain unchanged.

Code comes from GitHub; data, checkpoints, and final predictions live on Drive. Hot
paths are copied to the Colab SSD. Thresholds are selected only on validation; test is
evaluated once with the selected values. Re-running skips completed stages.

Expected on Drive:

    MyDrive/Thesis/
      00_data/U-DIADS-TL/
      80_models/
      99_evaluation/


## 1. Config


In [ ]:
REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = "/content/drive/MyDrive/Thesis"
REPO_DIR    = "/content/repo"
WORK_DIR    = "/content/work"

MODEL_ID = "facebook/mask2former-swin-tiny-coco-instance"
IMAGE_SIZE = 1152                 # maximum page side, matching the other models
SHORTEST_EDGE = 768               # U-DIADS pages are approximately 2:3
LONGEST_EDGE = IMAGE_SIZE
LATIN_EPOCHS = 100              # the already successful full-page baseline
TARGET_DENSE_UPDATES = 1000     # Latin2/Syr341 optimizer-update budget
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 1
LEARNING_RATE = 5e-5              # stable Mask2Former fine-tuning LR
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05
TRAIN_NUM_POINTS = 12544          # Hugging Face / Mask2Former default
MASK_LABEL_STRIDE = 1             # exact full-resolution GT masks
EVAL_EVERY = 10
SEED = 42
NUM_WORKERS = 2
ALLOW_RESOLUTION_FALLBACK = False # keep False for a comparable 1152 experiment
SMOKE = False                     # True -> one image, 1 epoch, one eval image

# Dense subsets are tiled so all stages retain the 100 learned task queries.
# With three training pages this gives 1002 and 1008 dense-subset updates.
RUNS = [
    dict(subset="Latin14396", queries=100, tiles=1, epochs=LATIN_EPOCHS,
         checkpoint_thresholds=(0.4, 0.3, 0.3)),
    dict(subset="Latin2", queries=100, tiles=2, epochs=None,
         checkpoint_thresholds=(0.8, 0.5, 0.3)),
    dict(subset="Syr341", queries=100, tiles=3, epochs=None,
         checkpoint_thresholds=(0.8, 0.5, 0.3)),
]

# Compact validation grid; tie-breaks are FM -> Line_IU -> Pixel_IU.
SCORE_THRESHOLDS = (0.4, 0.6, 0.8)
MASK_THRESHOLDS  = (0.3, 0.5)
NMS_THRESHOLDS   = (0.3, 0.7)
STITCH_SEAM_BAND_RATIO = 0.02   # tuned on validation only
STITCH_MIN_VERTICAL_OVERLAP = 0.5

import gc, json, math, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_ROOT, REPO_DIR, WORK_DIR = map(Path, (DRIVE_ROOT, REPO_DIR, WORK_DIR))
DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
DRIVE_EVAL   = DRIVE_ROOT / "99_evaluation"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"
LOCAL_EVAL   = WORK_DIR / "99_evaluation"
STAGE_DIR    = REPO_DIR / "50_modelling" / "03_instance_segmentation"

def run(cmd, cwd=None, check=True):
    printable = cmd if isinstance(cmd, str) else shlex.join(map(str, cmd))
    print(f"$ {printable}")
    return subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, check=check)

print("colab runtime:", IN_COLAB)


## 2. Mount Drive


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} not found -- create it first"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} not found -- upload 00_data there"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_EVAL.mkdir(parents=True, exist_ok=True)
print("drive OK:", DRIVE_ROOT)


## 3. Clone the repo and recreate its directory layout


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

for directory in (LOCAL_DATA, LOCAL_MODELS, LOCAL_EVAL):
    directory.mkdir(parents=True, exist_ok=True)

for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS),
                     ("99_evaluation", LOCAL_EVAL)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"{link.name:18} -> {target}")

SCRIPT = STAGE_DIR / "mask2former_udiads.py"
assert SCRIPT.is_file(), f"missing from clone: {SCRIPT}"


## 4. Copy U-DIADS-TL to the local SSD

The copy compares relative filenames and sizes, so re-running repairs an interrupted
transfer instead of accepting a half-populated directory.


In [ ]:
NEEDED = [
    "U-DIADS-TL/Latin14396", "U-DIADS-TL/Latin2", "U-DIADS-TL/Syr341",
    "U-DIADS-TL/coco_dataset_latin14396",
    "U-DIADS-TL/coco_dataset_latin2",
    "U-DIADS-TL/coco_dataset_syr341",
]

def file_manifest(root: Path):
    return {p.relative_to(root): p.stat().st_size for p in root.rglob("*") if p.is_file()}

def sync_data_dir(src: Path, dst: Path):
    assert src.is_dir(), f"missing on Drive: {src}"
    wanted = file_manifest(src)
    present = file_manifest(dst) if dst.is_dir() else {}
    todo = [rel for rel, size in wanted.items() if present.get(rel) != size]
    for rel in todo:
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src / rel, target)
    print(f"{src.relative_to(DRIVE_DATA)!s:48} {len(wanted):4} files, {len(todo):4} copied")

t0 = time.time()
for rel in NEEDED:
    sync_data_dir(DRIVE_DATA / rel, LOCAL_DATA / rel)
print(f"data ready in {time.time() - t0:.1f}s")


## 5. Dependencies and GPU check


In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r",
     str(STAGE_DIR / "requirements.txt")])
run("nvidia-smi || true", check=False)

import numpy as np
import pandas as pd
import torch, transformers

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
BF16 = torch.cuda.is_bf16_supported()
AMP_NAME = "bf16" if BF16 else "fp16 + GradScaler"
gpu = torch.cuda.get_device_properties(0)
print(f"torch={torch.__version__} transformers={transformers.__version__}")
print(f"GPU={gpu.name} VRAM={gpu.total_memory / 2**30:.1f} GiB AMP={AMP_NAME}")
assert transformers.__version__.startswith("5.14."), \
    "restart the runtime if pip replaced an already imported Transformers version"


## 6. Real 1152 resolution guard

This runs a real optimizer step for the heaviest sample in every configured stage,
including the memory-heavy Mask2Former criterion and full-resolution masks. By
default the notebook stops instead of silently lowering the 1152 maximum side.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("m2f_udiads", SCRIPT)
m2f = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m2f)

def probe_resolution(shortest_edge, longest_edge, cfg):
    processor = m2f.Mask2FormerImageProcessor.from_pretrained(MODEL_ID)
    dataset = m2f.CocoTextLineDataset(
        LOCAL_DATA / "U-DIADS-TL", cfg["subset"], "train", processor=processor,
        shortest_edge=shortest_edge, longest_edge=longest_edge, mask_label_stride=1,
        vertical_tiles=cfg["tiles"])
    heavy = None
    for index in range(len(dataset)):
        sample = dataset[index]
        if heavy is None or len(sample["class_labels"]) > len(heavy["class_labels"]):
            heavy = sample
    model = batch = inputs = loss = scaler = optimizer = None
    try:
        torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
        model = m2f.build_finetune_model(cfg["queries"], TRAIN_NUM_POINTS).cuda().train()
        batch = m2f.collate_mask2former([heavy])
        inputs = m2f.move_training_batch(batch, torch.device("cuda"))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                      weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler("cuda", enabled=not BF16)
        with m2f.autocast_context(torch.device("cuda"), True):
            loss = model(**inputs).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated() / 2**30
        shape = tuple(batch["pixel_values"].shape[-2:])
        count = len(batch["class_labels"][0])
        print(f"PASS {cfg['subset']} tiles={cfg['tiles']} "
              f"args={shortest_edge}/{longest_edge}, tensor={shape}, "
              f"instances={count}, peak={peak:.2f} GiB")
        budget = torch.cuda.get_device_properties(0).total_memory / 2**30 * 0.90
        if peak > budget:
            print(f"insufficient headroom: {peak:.2f} GiB > 90% budget {budget:.2f} GiB")
            return None
        return peak
    except torch.OutOfMemoryError:
        print(f"OOM args={shortest_edge}/{longest_edge}")
        return None
    finally:
        del model, batch, inputs, loss, scaler, optimizer, dataset, processor, heavy
        gc.collect(); torch.cuda.empty_cache()

candidates = [(768, 1152), (704, 1056), (640, 960)]
selected = None
for shortest, longest in candidates:
    peaks = [probe_resolution(shortest, longest, cfg) for cfg in RUNS]
    if all(peak is not None for peak in peaks):
        selected = (shortest, longest, max(peaks))
        break
    if not ALLOW_RESOLUTION_FALLBACK:
        raise RuntimeError(
            "Full-page 1152 does not fit this GPU. Use a high-memory Colab GPU, "
            "or explicitly set ALLOW_RESOLUTION_FALLBACK=True (results then are not "
            "directly comparable to the 1152 runs).")

assert selected is not None
SHORTEST_EDGE, LONGEST_EDGE, PROBE_PEAK_GIB = selected
print(f"selected max side={LONGEST_EDGE}; probe peak={PROBE_PEAK_GIB:.2f} GiB")


## 7. Restore known run directories from Drive

A finished model is identified by its configured epoch count plus `model.safetensors`. An
interrupted run is retrained because the compact local trainer intentionally stores
only the best model, not optimizer state.


In [ ]:
MODEL_REL_ROOT = Path("03_instance_segmentation/u-diads-tl")
EVAL_REL_ROOT = Path("03_instance_segmentation/u-diads-tl")
for cfg in RUNS:
    annotation_file = (LOCAL_DATA / "U-DIADS-TL" /
                       f"coco_dataset_{cfg['subset'].lower()}" / "train.json")
    train_pages = len(json.loads(annotation_file.read_text())["images"])
    if cfg["epochs"] is None:
        updates_per_epoch = math.ceil(train_pages * cfg["tiles"] / GRADIENT_ACCUMULATION)
        cfg["epochs"] = math.ceil(TARGET_DENSE_UPDATES / updates_per_epoch)
    cfg["optimizer_updates"] = (math.ceil(train_pages * cfg["tiles"] /
                                                GRADIENT_ACCUMULATION) * cfg["epochs"])
    epochs = 1 if SMOKE else cfg["epochs"]
    cfg["run_name"] = (f"m2f_swin_t_img{LONGEST_EDGE}_tiles{cfg['tiles']}_"
                       f"q{cfg['queries']}_{epochs}ep_bestfm_colab")
    cfg["tuned_name"] = (f"{cfg['run_name']}_stitched_tuned"
                         if cfg["tiles"] > 1 else f"{cfg['run_name']}_tuned")
    print(cfg["subset"], f"epochs={epochs}, updates={cfg['optimizer_updates']}")

def copy_tree_merge(src: Path, dst: Path):
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"restored {src} -> {dst}")

for cfg in RUNS:
    subset, name = cfg["subset"], cfg["run_name"]
    copy_tree_merge(DRIVE_MODELS / MODEL_REL_ROOT / subset / name,
                    LOCAL_MODELS / MODEL_REL_ROOT / subset / name)
    copy_tree_merge(DRIVE_EVAL / EVAL_REL_ROOT / subset / cfg["tuned_name"],
                    LOCAL_EVAL / EVAL_REL_ROOT / subset / cfg["tuned_name"])

def model_complete(path: Path):
    history = path / "history.csv"
    if not (path / "model.safetensors").is_file() or not history.is_file():
        return False
    rows = [line for line in history.read_text().splitlines() if line.strip()]
    expected_epoch = 1 if SMOKE else next(
        cfg["epochs"] for cfg in RUNS if path.name == cfg["run_name"])
    return bool(rows and rows[-1].split(",", 1)[0] == str(expected_epoch))


## 8. Train the full-page baseline and dense tiled models

Latin14396 is trained first on full pages. Latin2 and Syr341 both initialize directly
from that same 100-query checkpoint, then train on two and three column tiles. This
avoids randomly adding query slots and avoids propagating a weak Latin2 stage to Syr341.
The saved checkpoint maximizes validation FM at the subset-specific fixed thresholds.


In [ ]:
def sync_model(cfg):
    src = LOCAL_MODELS / MODEL_REL_ROOT / cfg["subset"] / cfg["run_name"]
    dst = DRIVE_MODELS / MODEL_REL_ROOT / cfg["subset"] / cfg["run_name"]
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("model synced:", dst)

latin_checkpoint = None
for cfg in RUNS:
    checkpoint = LOCAL_MODELS / MODEL_REL_ROOT / cfg["subset"] / cfg["run_name"]
    if model_complete(checkpoint):
        print(f"[skip complete] {cfg['subset']}: {checkpoint}")
    else:
        if checkpoint.exists():
            print("removing incomplete local run:", checkpoint)
            shutil.rmtree(checkpoint)
        cmd = [sys.executable, str(SCRIPT), "train", "--subsets", cfg["subset"],
               "--epochs", str(1 if SMOKE else cfg["epochs"]),
               "--batch-size", str(BATCH_SIZE),
               "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
               "--learning-rate", str(LEARNING_RATE),
               "--weight-decay", str(WEIGHT_DECAY),
               "--warmup-ratio", str(WARMUP_RATIO),
               "--num-queries", str(cfg["queries"]),
               "--train-vertical-tiles", str(cfg["tiles"]),
               "--train-num-points", str(TRAIN_NUM_POINTS),
               "--mask-label-stride", str(MASK_LABEL_STRIDE),
               "--shortest-edge", str(SHORTEST_EDGE),
               "--longest-edge", str(LONGEST_EDGE),
               "--eval-every", str(1 if SMOKE else EVAL_EVERY),
               "--checkpoint-metric", "val_fm",
               "--checkpoint-score-threshold", str(cfg["checkpoint_thresholds"][0]),
               "--checkpoint-mask-threshold", str(cfg["checkpoint_thresholds"][1]),
               "--checkpoint-nms-threshold", str(cfg["checkpoint_thresholds"][2]),
               "--stitch-seam-band-ratio", str(STITCH_SEAM_BAND_RATIO),
               "--stitch-min-vertical-overlap", str(STITCH_MIN_VERTICAL_OVERLAP),
               "--workers", str(NUM_WORKERS),
               "--seed", str(SEED), "--run-name", cfg["run_name"]]
        if cfg["subset"] != "Latin14396":
            assert latin_checkpoint is not None, "Latin14396 checkpoint must run first"
            cmd += ["--init-checkpoint", str(latin_checkpoint)]
        if cfg["tiles"] > 1:
            cmd += ["--stitch-tile-instances"]
        if SMOKE:
            cmd += ["--max-train-images", "1", "--max-eval-images", "1"]
        run(cmd, cwd=str(REPO_DIR))
        assert model_complete(checkpoint), f"training did not complete: {checkpoint}"
        sync_model(cfg)
    if cfg["subset"] == "Latin14396":
        latin_checkpoint = checkpoint

print("all model stages complete")


## 9. Validation-only threshold selection and one test evaluation

Each grid point sees only the validation split. The test split is run once after the
best `(score, mask, NMS)` tuple is fixed. Dense-subset predictions also use seam
stitching with parameters fixed from validation. Validation instance PNGs stay on the local
SSD; only compact validation summaries and the complete tuned test directory are
copied to Drive.


In [ ]:
def token(value):
    return str(value).replace(".", "p")

def evaluate(cfg, split, score, mask, nms, run_name):
    checkpoint = LOCAL_MODELS / MODEL_REL_ROOT / cfg["subset"] / cfg["run_name"]
    cmd = [sys.executable, str(SCRIPT), "evaluate", "--subsets", cfg["subset"],
           "--checkpoint", str(checkpoint), "--eval-split", split,
           "--shortest-edge", str(SHORTEST_EDGE),
           "--longest-edge", str(LONGEST_EDGE),
           "--vertical-tiles", str(cfg["tiles"]),
           "--stitch-seam-band-ratio", str(STITCH_SEAM_BAND_RATIO),
           "--stitch-min-vertical-overlap", str(STITCH_MIN_VERTICAL_OVERLAP),
           "--score-threshold", str(score), "--mask-threshold", str(mask),
           "--nms-threshold", str(nms), "--run-name", run_name]
    if cfg["tiles"] > 1:
        cmd += ["--stitch-tile-instances"]
    if SMOKE:
        cmd += ["--max-eval-images", "1"]
    run(cmd, cwd=str(REPO_DIR))
    summary = LOCAL_EVAL / EVAL_REL_ROOT / cfg["subset"] / run_name / "summary.json"
    return json.loads(summary.read_text())

final_rows = []
for cfg in RUNS:
    tuned_name = cfg["tuned_name"]
    tuned_dir = LOCAL_EVAL / EVAL_REL_ROOT / cfg["subset"] / tuned_name
    if (tuned_dir / "summary.json").is_file():
        print(f"[skip tuned test] {cfg['subset']}")
        final_rows.append(json.loads((tuned_dir / "summary.json").read_text()))
        continue

    candidates = []
    for score in SCORE_THRESHOLDS:
        for mask in MASK_THRESHOLDS:
            for nms in NMS_THRESHOLDS:
                stitch_tag = "stitched_" if cfg["tiles"] > 1 else ""
                val_name = (f"{cfg['run_name']}__{stitch_tag}val_s{token(score)}_"
                            f"m{token(mask)}_n{token(nms)}")
                summary_path = (LOCAL_EVAL / EVAL_REL_ROOT / cfg["subset"] /
                                val_name / "summary.json")
                if summary_path.is_file():
                    summary = json.loads(summary_path.read_text())
                else:
                    summary = evaluate(cfg, "val", score, mask, nms, val_name)
                candidates.append(summary)

    best = max(candidates, key=lambda row: (row["FM"], row["Line_IU"], row["Pixel_IU"]))
    print(f"BEST VAL {cfg['subset']}: score={best['score_threshold']} "
          f"mask={best['mask_threshold']} nms={best['nms_threshold']} FM={best['FM']:.4f}")
    final = evaluate(cfg, "test", best["score_threshold"], best["mask_threshold"],
                     best["nms_threshold"], tuned_name)
    final["validation_selected_FM"] = best["FM"]
    (tuned_dir / "summary.json").write_text(json.dumps(final, indent=2))
    final_rows.append(final)

    drive_tuned = DRIVE_EVAL / EVAL_REL_ROOT / cfg["subset"] / tuned_name
    shutil.copytree(tuned_dir, drive_tuned, dirs_exist_ok=True)
    grid_suffix = "stitched_val_grid" if cfg["tiles"] > 1 else "val_grid"
    val_out = DRIVE_EVAL / EVAL_REL_ROOT / cfg["subset"] / f"{cfg['run_name']}_{grid_suffix}"
    val_out.mkdir(parents=True, exist_ok=True)
    for row in candidates:
        name = (f"s{token(row['score_threshold'])}_m{token(row['mask_threshold'])}_"
                f"n{token(row['nms_threshold'])}.json")
        (val_out / name).write_text(json.dumps(row, indent=2))
    print("evaluation synced:", drive_tuned)


## 10. Final table and aggregate JSON


In [ ]:
columns = ["subset", "pages", "mean_instances", "score_threshold",
           "mask_threshold", "nms_threshold", "Pixel_IU", "Line_IU",
           "DR", "RA", "FM"]
frame = pd.DataFrame(final_rows)[columns]
display(frame.style.format({c: "{:.4f}" for c in columns[2:]}))
print("macro mean:")
display(frame[["Pixel_IU", "Line_IU", "DR", "RA", "FM"]].mean().to_frame("mean").T)

aggregate = LOCAL_EVAL / EVAL_REL_ROOT / f"mask2former_tiled_img{LONGEST_EDGE}_stitched_colab.json"
aggregate.parent.mkdir(parents=True, exist_ok=True)
aggregate.write_text(json.dumps(final_rows, indent=2))
drive_aggregate = DRIVE_EVAL / EVAL_REL_ROOT / aggregate.name
drive_aggregate.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(aggregate, drive_aggregate)
print("aggregate:", drive_aggregate)
